# Exercise 1: Compare NumPy and CuPy transfer paths

## Question

For the same PyMUST raw-RF phantom workload and the same CUDA beamforming engine, how does passing host NumPy arrays compare with keeping inputs GPU-resident through CuPy?

## Hypothesis

Both paths use the same CUDA beamforming kernel from `mach-beamform` 0.2.1. Its NumPy path automatically stages CPU arrays to the GPU for each call: it allocates temporary device buffers, copies IQ and geometry inputs, then copies a NumPy output back to the host. I expect CuPy to reduce per-call wall latency by uploading the static geometry once, uploading the IQ block once per call, and keeping mach's beamformed output plus the Power Doppler reduction on the GPU until the final map is copied back. I expect numerically comparable maps. This predicts a benefit from the full array-placement path, including output and reduction placement; it does not predict that mach's kernel itself is faster with CuPy. The size of the benefit is uncertain until we measure individual transfers and setup costs.

## What this experiment changes

Both modes call the same `mach` CUDA beamformer. In the NumPy path, `mach` detects CPU arrays and stages IQ, receiver positions, scan points, and transmit arrival times to temporary GPU buffers on each call; because the output follows the NumPy input namespace, `mach` copies beamformed output back to host memory, and this project's Power Doppler reduction runs with NumPy. In the CuPy path, this project copies the three geometry arrays to GPU once, uploads the IQ block once per call, and passes CUDA-resident arrays to `mach`. The beamformed output and reduction stay on the GPU; only the final Power Doppler map is copied back. So the experiment compares the complete host-array and GPU-resident paths through the same CUDA kernel, including staging, output placement, and reduction placement. `mach`'s v0.2.1 implementation is described in its [Python wrapper](https://github.com/Forest-Neurotech/mach/blob/v0.2.1/src/mach/kernel.py#L208-L285) and [CUDA binding](https://github.com/Forest-Neurotech/mach/blob/v0.2.1/src/mach/kernel.cu#L1073-L1215).


### Where each stage runs

| Stage | NumPy host-array path | CuPy GPU-resident path |
|---|---|---|
| RF → IQ | CPU: PyMUST converts the raw RF array before the benchmark loop. | CPU: same conversion, before the benchmark loop. |
| IQ and geometry placement | Host arrays go to `mach`; it stages IQ and geometry to temporary GPU buffers on each call. | Geometry is copied to the GPU once and retained; each call copies that ensemble's IQ to the GPU. |
| Beamforming | Same CUDA beamformer in `mach`. | Same CUDA beamformer in `mach`. |
| Power Doppler reduction | Beamformed output returns to host memory; NumPy reduces it on the CPU. | Reduction stays on the GPU; only the final map is copied to host memory. |

“Geometry” here means the receiver-element positions, scan-grid positions, and transmit-wave arrival times used by delay-and-sum beamforming. In CuPy mode, `cp.asarray` copies those three arrays from host memory to GPU memory once per prepared geometry, then the arrays are reused across every warmup and measured ensemble. That one-time cost is amortized over later ensembles while probe and scan settings remain fixed; it is paid again after a process/runtime restart or geometry change. IQ channel data is separate and is uploaded for every CuPy call.

CuPy does **not** cache the IQ block between calls: every `run_ensemble` call uploads its IQ input. Geometry is different: the three CuPy geometry arrays are held by the prepared-geometry object and reused for warmups and measured calls, including calls on later, distinct ensembles, while the probe and scan configuration stay fixed. In NumPy mode, this project makes no explicit geometry upload, but `mach` stages the host geometry on each beamforming call.

For a live stream, each distinct RF ensemble goes through CPU RF-to-IQ conversion; CuPy then uploads each new IQ block, while the fixed geometry stays on the GPU across ensembles.

The input is the PyMUST rotating-disk tissue-mimicking phantom, not a brain fUSI recording. The benchmark reuses the same 32-frame ensemble across timing samples to estimate latency variation. A live stream normally processes new RF data; this repeated-input measurement does not model its acquisition queue or per-ensemble RF-to-IQ cost.

## Plan

1. Run both modes on the same Colab T4 with the same dataset, scan grid, batch size, warmup count, and measured run count.
2. Check that the reports describe the same input and runtime, then compare the output maps numerically and visually.
3. Compare p50, p95, p99, and maximum ensemble wall latency, plus the first-sample-to-output estimate.
4. Interpret the result against the hypothesis. The pinned `mach` source predicts per-call CPU-array staging and output copyback on the NumPy path; use a short Nsight Systems trace to confirm runtime copy counts, sizes, and time before assigning the measured gain to particular transfers.

The run below uses 200 measured ensembles after 10 warmups, matching the exercise procedure. The short two-run report from the earlier environment smoke check is retained separately and is not used as comparison data.


## Prepare a fresh Colab runtime

Connect this notebook to a Colab GPU kernel in VS Code. In the Colab **Contents** view, upload the current local `realtime-beamforming-lab` folder so the runtime receives the working-tree code and notebook. A fresh runtime will not have the previous checkout or reports.

The project directory must be `/content/realtime-beamforming-lab`. The next cell checks that the uploaded checkout includes the benchmark package.


In [ ]:
from pathlib import Path
import os

repo = Path("/content/realtime-beamforming-lab")
assert (repo / "pyproject.toml").is_file(), (
    f"Upload the local project folder to Colab Contents; expected {repo}"
)
assert (repo / "src/beamforming_lab/benchmark.py").is_file(), "The uploaded checkout is incomplete"
os.chdir(repo)
print(f"Using checkout: {repo}")


In [ ]:
!nvidia-smi


## Install the Colab GPU dependencies

Run this once in a fresh runtime. It installs the uploaded local checkout and the Colab-compatible CuPy wheel. When it finishes, restart the notebook kernel, then continue with the verification cell. The comparison itself must not start until the CuPy and GPU checks pass.


In [ ]:
%pip uninstall -y cupy-cuda12x cupy-cuda13x
%pip install --no-cache-dir -e '.[gpu-colab]'


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import os
import cupy as cp

repo = Path("/content/realtime-beamforming-lab")
assert (repo / "pyproject.toml").is_file(), "The uploaded project checkout is missing after kernel restart"
os.chdir(repo)
cupy_wheels = sorted(
    dist.metadata.get("Name", "")
    for dist in metadata.distributions()
    if dist.metadata.get("Name", "").lower().startswith("cupy-cuda")
)
print("CuPy module:", cp.__file__)
print("CuPy wheels:", cupy_wheels)
print("CUDA devices:", cp.cuda.runtime.getDeviceCount())
print("CUDA runtime:", cp.cuda.runtime.runtimeGetVersion())
assert cp.cuda.runtime.getDeviceCount() > 0
assert len(cupy_wheels) == 1 and cupy_wheels[0].lower() == "cupy-cuda13x", cupy_wheels
cp.cuda.Device().synchronize()


## Fix the comparison conditions

We keep the default 251 × 251 scan grid and process all 32 frames together (`batch_size=32`), so each timed call to `mach` handles one 32-frame IQ ensemble. Each mode gets one separately recorded cold first call, 10 warmup calls, and 200 measured calls on the same preconverted IQ block. The repeated input controls the workload while estimating timing variation; it is not a cached GPU input, and the CuPy path still uploads IQ on every call. In a live stream, the IQ values change each ensemble, while fixed geometry can remain GPU-resident. The matched settings make the steady-state per-call distributions comparable.

The two small JSON reports go in `results/`. The Power Doppler maps go in `artifacts/` because they are intermediate arrays used by the notebook's correctness analysis.


In [ ]:
from pathlib import Path

RUNS = 200
WARMUP = 10
BATCH_SIZE = 32
GRID_SIZE = 251

results_dir = repo / "results"
artifacts_dir = repo / "artifacts"
results_dir.mkdir(exist_ok=True)
artifacts_dir.mkdir(exist_ok=True)

report_paths = {
    "numpy": results_dir / "exercise1-pymust-numpy.json",
    "cupy": results_dir / "exercise1-pymust-cupy.json",
}
power_paths = {
    "numpy": artifacts_dir / "exercise1-pymust-numpy-power.npy",
    "cupy": artifacts_dir / "exercise1-pymust-cupy-power.npy",
}
print(f"Conditions: {RUNS} measured runs, {WARMUP} warmups, batch size {BATCH_SIZE}, grid {GRID_SIZE} × {GRID_SIZE}")


## Run the matched benchmarks

Each command starts from the same cached raw dataset and constructs the same workload. Dataset loading and CPU RF-to-IQ conversion happen before the timing loop. With `--measure-cold-start`, the command records one separate first-result measurement before warmups: NumPy's first CUDA use occurs inside its first `mach` call; CuPy's occurs while preparing geometry. CuPy geometry is then reused; every CuPy ensemble call still uploads its IQ input. Each NumPy call passes host IQ and geometry to `mach`, which stages CPU arrays to the GPU and returns a NumPy beamformed output. The steady-state pipeline timer includes those internal `mach` transfers and the mode-specific Power Doppler reduction. The explicit upload timer isolates only the CuPy IQ upload. We save each mode's Power Doppler map so correctness is checked before interpreting speed.

This cell takes longer than the smoke check because it measures 400 ensembles, 20 warmups, and one extra cold call per mode. NumPy and CuPy run as separate `beamform-lab` subprocesses, giving each a fresh process-level CUDA context. The cold metric is reported separately from the 200-call steady-state distribution.


In [ ]:
import json
import subprocess

for mode in ("numpy", "cupy"):
    command = [
        "beamform-lab", "benchmark",
        "--dataset", "pymust",
        "--mode", mode,
        "--measure-cold-start",
        "--runs", str(RUNS),
        "--warmup", str(WARMUP),
        "--batch-size", str(BATCH_SIZE),
        "--grid-size", str(GRID_SIZE),
        "--output", str(report_paths[mode]),
        "--save-power", str(power_paths[mode]),
    ]
    print("\nRunning:", " ".join(command), flush=True)
    completed = subprocess.run(command, cwd=repo, text=True, capture_output=True)
    if completed.returncode:
        print(completed.stdout[-4000:])
        print(completed.stderr[-4000:])
        completed.check_returncode()
    report = json.loads(report_paths[mode].read_text())
    pipeline = report["ensemble_pipeline_wall_ms"]
    first_sample = report["first_sample_to_output_ms"]
    cold = report["cold_start_measurement"]
    print(
        f"{mode}: GPU={report['runtime_provenance']['gpu']['name']}; "
        f"pipeline p50={pipeline['p50']:.2f} ms, p95={pipeline['p95']:.2f} ms; "
        f"warm first-sample estimate p50={first_sample['p50']:.2f} ms; "
        f"cold setup + first ensemble={cold['device_setup_plus_first_ensemble_wall_ms']:.2f} ms; "
        f"cold first-sample estimate={cold['first_sample_to_output_ms']:.2f} ms; "
        f"report={report_paths[mode].name}"
    )


## Check that this is a fair pair, then compare outputs

Before comparing times, verify that both reports used the same raw-RF fingerprint, grid, batch size, run counts, and GPU. Then compare the saved maps with normalized L2 error and correlation. This checks that the different transfer and reduction paths preserve the image result within floating-point tolerance; it does not require bitwise equality.


In [ ]:
import json
import numpy as np

reports = {mode: json.loads(report_paths[mode].read_text()) for mode in ("numpy", "cupy")}
power_maps = {mode: np.load(power_paths[mode]) for mode in ("numpy", "cupy")}

numpy_report, cupy_report = reports["numpy"], reports["cupy"]
assert numpy_report["data_provenance"]["rf_array_sha256"] == cupy_report["data_provenance"]["rf_array_sha256"]
assert numpy_report["scan_grid_points"] == cupy_report["scan_grid_points"]
assert numpy_report["batch_size_frames"] == cupy_report["batch_size_frames"] == BATCH_SIZE
assert numpy_report["measurement_runs"] == cupy_report["measurement_runs"] == RUNS
assert numpy_report["warmup_runs"] == cupy_report["warmup_runs"] == WARMUP
assert numpy_report["runtime_provenance"]["gpu"]["name"] == cupy_report["runtime_provenance"]["gpu"]["name"]
assert power_maps["numpy"].shape == power_maps["cupy"].shape

numpy_map = power_maps["numpy"].astype(np.float64)
cupy_map = power_maps["cupy"].astype(np.float64)
difference = cupy_map - numpy_map
scale = max(float(np.linalg.norm(numpy_map.ravel())), np.finfo(np.float64).tiny)
relative_l2_error = float(np.linalg.norm(difference.ravel()) / scale)
shared_peak = max(float(np.max(numpy_map)), float(np.max(cupy_map)), np.finfo(np.float64).tiny)
max_abs_error = float(np.max(np.abs(difference)))
peak_relative_error = max_abs_error / shared_peak
correlation = float(np.corrcoef(numpy_map.ravel(), cupy_map.ravel())[0, 1])

print(f"Output shape: {numpy_map.shape}")
print(f"Mean Power Doppler feature — NumPy: {numpy_report['mean_output_feature']:.6g}; CuPy: {cupy_report['mean_output_feature']:.6g}")
print(f"Normalized L2 map error: {relative_l2_error:.3e}")
print(f"Maximum absolute map error: {max_abs_error:.6g} ({peak_relative_error:.3e} of shared peak)")
print(f"Pixel-wise map correlation: {correlation:.8f}")

print("\nTiming context:")
for mode in ("numpy", "cupy"):
    report = reports[mode]
    print(f"{mode}: RF-to-IQ once={report['rf_to_iq_ms_once']:.2f} ms; "
          f"explicit geometry upload once={report['static_geometry_upload_ms_once']:.2f} ms")
cupy_upload = reports["cupy"]["per_ensemble_input_upload_ms"]
print(f"CuPy explicit IQ upload: p50={cupy_upload['p50']:.2f} ms, p95={cupy_upload['p95']:.2f} ms per ensemble")
print("NumPy host-array transfers are handled inside mach and are not isolated by this upload timer.")


In [ ]:
import matplotlib.pyplot as plt

shared_peak = max(float(np.max(numpy_map)), float(np.max(cupy_map)), np.finfo(np.float64).tiny)
floor = 1e-4
numpy_db = 10 * np.log10(np.maximum(numpy_map / shared_peak, floor))
cupy_db = 10 * np.log10(np.maximum(cupy_map / shared_peak, floor))
abs_difference_percent = 100 * np.abs(difference) / shared_peak
diff_vmax = max(float(np.percentile(abs_difference_percent, 99.9)), 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for ax, image, title in [
    (axes[0], numpy_db, "NumPy host arrays"),
    (axes[1], cupy_db, "CuPy GPU-resident path"),
]:
    view = ax.imshow(image.T, origin="lower", cmap="magma", vmin=-40, vmax=0, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Lateral grid index")
    ax.set_ylabel("Depth grid index")
fig.colorbar(view, ax=axes[:2], label="Power Doppler (dB relative to shared peak)", shrink=0.85)

error_view = axes[2].imshow(
    abs_difference_percent.T, origin="lower", cmap="viridis", vmin=0, vmax=diff_vmax, aspect="auto"
)
axes[2].set_title("Absolute difference")
axes[2].set_xlabel("Lateral grid index")
axes[2].set_ylabel("Depth grid index")
fig.colorbar(error_view, ax=axes[2], label="% of shared peak (color clipped at p99.9)", shrink=0.85)
fig.suptitle(
    f"Power Doppler output agreement · normalized L2 error {relative_l2_error:.2e} · correlation {correlation:.6f}",
    fontsize=14,
)
output_plot_path = results_dir / "exercise1-pymust-output-comparison.png"
fig.savefig(output_plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {output_plot_path}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

percentiles = ("p50", "p95", "p99", "max")
mode_labels = {"numpy": "NumPy host arrays", "cupy": "CuPy GPU-resident"}
colors = {"numpy": "#D87532", "cupy": "#2867A5"}
metric_specs = [
    ("ensemble_pipeline_wall_ms", "Per-ensemble pipeline wall time"),
    ("first_sample_to_output_ms", "First-sample-to-output estimate"),
]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True, constrained_layout=True)
x = np.arange(len(percentiles))
width = 0.36
for ax, (metric_key, title) in zip(axes, metric_specs):
    for offset, mode in ((-width / 2, "numpy"), (width / 2, "cupy")):
        values = [reports[mode][metric_key][key] for key in percentiles]
        bars = ax.bar(x + offset, values, width, label=mode_labels[mode], color=colors[mode])
        ax.bar_label(bars, fmt="%.0f", padding=2, fontsize=8)
    ax.set_title(title)
    ax.set_xticks(x, percentiles)
    ax.set_ylabel("Latency (ms)")
    ax.grid(axis="y", alpha=0.22)
    ax.set_axisbelow(True)
    ax.legend(frameon=False)

numpy_p50 = reports["numpy"]["ensemble_pipeline_wall_ms"]["p50"]
cupy_p50 = reports["cupy"]["ensemble_pipeline_wall_ms"]["p50"]
cupy_speedup = numpy_p50 / cupy_p50 if cupy_p50 else float("nan")
gpu_name = reports["cupy"]["runtime_provenance"]["gpu"]["name"]
fig.suptitle(
    f"Exercise 1 · {gpu_name} · {RUNS} runs/mode · {WARMUP} warmups · batch {BATCH_SIZE} · CuPy p50 speedup {cupy_speedup:.2f}×",
    fontsize=13,
)
timing_plot_path = results_dir / "exercise1-pymust-latency-comparison.png"
fig.savefig(timing_plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {timing_plot_path}")
print(f"Ensemble p50: NumPy {numpy_p50:.2f} ms; CuPy {cupy_p50:.2f} ms; NumPy/CuPy ratio {cupy_speedup:.2f}×")


### Cold start versus prepared first-sample estimate

The last plot compares the measured cold first result with the warmed/prepared p50 estimate. The cold timing starts before geometry preparation and ends when the first Power Doppler map is ready; the report then adds CPU RF-to-IQ time and the estimated acquisition window. The warm estimate uses the warmed per-ensemble p50 with those same stages and assumes CUDA and CuPy geometry are ready. Both are serial estimates, not end-to-end capture measurements. There is one cold observation per mode, so repeat fresh processes to measure startup variation.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

modes = ("numpy", "cupy")
labels = {"numpy": "NumPy host arrays", "cupy": "CuPy GPU-resident"}
colors = {"numpy": "#D87532", "cupy": "#2867A5"}
if not all(reports[mode].get("cold_start_measurement") is not None for mode in modes):
    raise RuntimeError("Rerun both benchmark commands with --measure-cold-start before plotting.")

warm_estimates = [
    reports[mode]["first_sample_to_output_ms"]["p50"] for mode in modes
]
cold_estimates = [
    reports[mode]["cold_start_measurement"]["first_sample_to_output_ms"]
    for mode in modes
]
x = np.arange(len(modes))
width = 0.34
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
warm_bars = ax.bar(
    x - width / 2,
    warm_estimates,
    width,
    label="Warm/prepared estimate (p50)",
    color="#8B9AA8",
)
cold_bars = ax.bar(
    x + width / 2,
    cold_estimates,
    width,
    label="Cold first-result estimate (one observation)",
    color=[colors[mode] for mode in modes],
)
ax.bar_label(warm_bars, fmt="%.0f ms", padding=2, fontsize=8)
ax.bar_label(cold_bars, fmt="%.0f ms", padding=2, fontsize=8)
ax.set_xticks(x, [labels[mode] for mode in modes])
ax.set_ylabel("Serial first-sample-to-output estimate (ms)")
ax.set_title("Cold first result versus warmed/prepared estimate")
ax.grid(axis="y", alpha=0.22)
ax.set_axisbelow(True)
ax.legend(frameon=False)
ax.text(
    0.01,
    0.99,
    "Cold timing includes first device setup and the first map.\n"
    "Warm timing assumes CUDA and CuPy geometry are ready.\n"
    "Repeat fresh processes to characterize cold-start variation.",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=8,
    bbox={"facecolor": "white", "alpha": 0.84, "edgecolor": "none"},
)
plot_path = results_dir / "exercise1-pymust-cold-start.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")


## Interpret this run

### Read the measurements

Use the printed values and plots above to answer three questions:

1. **Were the inputs and outputs comparable?** The check cell confirms matching raw-RF fingerprints, grid, batch size, run counts, and GPU. The map comparison reports normalized L2 error, maximum difference, and pixel correlation.
2. **How did warmed latency compare?** Read p50 and tail values from the latency plot. A lower CuPy wall time is consistent with the hypothesis that GPU-resident arrays avoid some staging and host round trips, but the benchmark does not isolate the time of every NumPy-path transfer.
3. **What did first use cost?** The last plot puts the single cold first-result estimate beside a warm/prepared p50 estimate. Cold timing includes device setup and the first map. Warm timing assumes CUDA and CuPy geometry are ready. Both add CPU RF-to-IQ time and an estimated acquisition window serially; neither measures live capture-to-display latency.

Both modes use the CUDA beamformer in `mach`. RF-to-IQ conversion is CPU-side in both. NumPy passes host arrays to `mach`, so its internal staging and output copyback are included in the timed call; this report does not split their individual costs. CuPy keeps geometry resident across calls but uploads IQ for each call and keeps the reduction on GPU. Its explicit IQ-upload timer does not make the NumPy transfers separately measurable. Use Nsight Systems to verify and time those copies before assigning the observed difference to a particular transfer.

There is one cold observation per mode, so treat that comparison as a first measurement, not a stable startup estimate. Repeat in several fresh processes and alternate mode order to characterize variation. The benchmark reuses one phantom IQ ensemble for timing percentiles; it does not model RF-to-IQ for a sequence of new live ensembles.

### Next steps

1. Repeat the paired benchmark in alternating order across fresh processes on a fixed Linux NVIDIA instance.
2. Capture a separate short Nsight Systems trace there to check H2D/D2H copy counts, sizes, timing, synchronization, and launch gaps.
3. Then benchmark distinct RF ensembles through RF-to-IQ, transfer, beamforming, and reduction. This rotating-disk phantom is a pipeline workload, not a brain fUSI recording or a live-stream latency result.
